# EXPERIMENTAL VERSION

# Beware of the Dog: Assessing the Impact of Data Augmentation on Dog-Breed Identification

**Deep Learning 25/26**

- Francisco Guimarães
- Jorge Caldeira
- Sofia Reia

In [ ]:
!pip install grad-cam torchmetrics

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
import gc

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torchmetrics

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# GLOBAL SEED
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
TRAIN_DIR = '/content/train_local/train/'
TEST_DIR = '/content/test_local/test/'
LABELS_CSV = 'labels.csv'

labels_df = pd.read_csv(LABELS_CSV) if os.path.exists(LABELS_CSV) else pd.DataFrame(columns=['id', 'breed'])
breed_list = sorted(labels_df['breed'].unique()) if not labels_df.empty else []
breed_to_idx = {breed: i for i, breed in enumerate(breed_list)}

from sklearn.model_selection import train_test_split
if not labels_df.empty:
    train_df, val_df = train_test_split(labels_df, test_size=0.2, random_state=42, stratify=labels_df['breed'])
else:
    train_df, val_df = pd.DataFrame(), pd.DataFrame()

class DogDataset(Dataset):
    def __init__(self, dataframe, img_dir, transform=None):
        self.dataframe = dataframe
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_id = self.dataframe.iloc[idx, 0]
        breed_name = self.dataframe.iloc[idx, 1]
        label = breed_to_idx.get(breed_name, 0)

        img_path = os.path.join(self.img_dir, f"{img_id}.jpg")
        try:
            image = Image.open(img_path).convert("RGB")
        except:
            image = Image.new('RGB', (224, 224))
            
        if self.transform:
            image = self.transform(image)

        return image, label, img_path


In [ ]:
imagenet_normalization = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

baseline_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    imagenet_normalization
])

augmented_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(45),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    imagenet_normalization
])


In [ ]:
def initialize_model(model_name="resnet50", num_classes=120, mode="freeze"):
    if model_name == "resnet50":
        model = models.resnet50(weights='DEFAULT')

    if mode == "freeze":
        print("Setup: RESNET50 (FREEZE)")
        for param in model.parameters():
            param.requires_grad = False
        
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

    elif mode == "partial_finetune":
        print("Setup: RESNET50 (PARTIAL_FINETUNE)")
        for param in model.parameters():
            param.requires_grad = False

        for param in model.layer4.parameters():
            param.requires_grad = True
            
        # FIX: Not forcing BN layers to eval during partial finetuning so they can update stats

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, num_classes)
    model.fc.weight.requires_grad = True
    model.fc.bias.requires_grad = True

    return model


In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels, paths in dataloader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = (correct / total) * 100
    return epoch_loss, epoch_acc

# FIix for Metric Instability/Confusion Matrix
def validate_model(model, loader, criterion, device, num_classes=120):
    model.eval()
    val_loss = 0.0
    total = 0
    
    # Using TorchMetrics iteratively to try to prevent OOM errors
    metric_f1 = torchmetrics.F1Score(task="multiclass", num_classes=num_classes, average="macro").to(device)
    metric_acc_top1 = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes, top_k=1).to(device)
    metric_acc_top5 = torchmetrics.Accuracy(task="multiclass", num_classes=num_classes, top_k=5).to(device)
    metric_cm = torchmetrics.ConfusionMatrix(task="multiclass", num_classes=num_classes).to(device)

    misclassified_images = []
    correct_images = []

    with torch.no_grad():
        for images, labels, paths in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            total += labels.size(0)

            metric_f1.update(outputs, labels)
            metric_acc_top1.update(outputs, labels)
            metric_acc_top5.update(outputs, labels)
            metric_cm.update(outputs, labels)
            
            _, predicted = outputs.max(1)
            for i in range(len(labels)):
                true_label = labels[i].item()
                predicted_label = predicted[i].item()
                if true_label != predicted_label:
                    misclassified_images.append({"path": paths[i], "true": true_label, "pred": predicted_label})
                else:
                    correct_images.append({"path": paths[i], "true": true_label, "pred": predicted_label})

    avg_loss = val_loss / total
    acc_top1 = metric_acc_top1.compute().item() * 100.
    acc_top5 = metric_acc_top5.compute().item() * 100.
    macro_f1 = metric_f1.compute().item()
    cm = metric_cm.compute().cpu().numpy()
    
    metric_f1.reset()
    metric_acc_top1.reset()
    metric_acc_top5.reset()
    metric_cm.reset()

    correct_count = np.trace(cm)
    incorrect_count = np.sum(cm) - correct_count
    accuracy_breakdown = np.array([[correct_count, incorrect_count]])
    
    #fallback if breed_list is empty during testing
    safe_index = breed_list if len(breed_list) == num_classes else range(num_classes)
    matrix_df = pd.DataFrame(cm, index=safe_index, columns=safe_index)

    return avg_loss, acc_top1, acc_top5, macro_f1, cm, accuracy_breakdown, matrix_df, misclassified_images, correct_images


In [ ]:
# Fix for Grad-CAM Hardcoding
def get_gradcam(img_path, true_label, pred_label, model, transform_pipeline, target_layer=None):
    if target_layer is None:
        target_layer = model.layer4[-1] # Can be configured now (tldr we can try to use InceptionV3)
        
    img = cv2.imread(img_path)
    if img is None: return
        
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    
    pil_img = Image.fromarray(img)
    tensor_img = transform_pipeline(pil_img).unsqueeze(0).to(device)
    tensor_img.requires_grad = True

    _, _, h, w = tensor_img.shape
    img_resized = cv2.resize(img, (w, h))
    img_rgb = img_resized.astype(np.float32) / 255.0

    target_layers = [target_layer]
    cam = GradCAM(model=model, target_layers=target_layers)
    targets = [ClassifierOutputTarget(int(pred_label))]
    
    grayscale_cam = cam(input_tensor=tensor_img, targets=targets)[0]
    visualization = show_cam_on_image(img_rgb, grayscale_cam, use_rgb=True)

    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1)
    plt.imshow(img_resized)
    lbl_t = breed_list[true_label] if breed_list and true_label < len(breed_list) else true_label
    plt.title(f"True: {lbl_t}")
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.imshow(visualization)
    lbl_p = breed_list[pred_label] if breed_list and pred_label < len(breed_list) else pred_label
    plt.title(f"Pred: {lbl_p}")
    plt.axis("off")

    plt.show()


In [ ]:
# Fix for repeated code + unsafe checkpointing
def run_experiment(exp_name, mode, train_loader, val_loader, val_transform, num_epochs=20):
    print(f"\n{'='*50}\nExperiment: {exp_name}\n{'='*50}")
    
    num_classes = len(breed_list) if breed_list else 120
    model = initialize_model("resnet50", num_classes=num_classes, mode=mode).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.0001)

    train_losses, val_losses, train_accs, val_accs_top1, val_accs_top5, val_f1_history = [], [], [], [], [], []
    best_val_acc = 0.0
    checkpoint_path = f"best_model_{exp_name.replace(' ', '_').lower()}.pth"

    for epoch in range(num_epochs):
        t_loss, t_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        v_loss, v_top1, v_top5, v_f1, cm, acc_breakdown, cm_df, misclassified, correct = validate_model(
            model, val_loader, criterion, device, num_classes=num_classes
        )

        train_losses.append(t_loss)
        val_losses.append(v_loss)
        train_accs.append(t_acc)
        val_accs_top1.append(v_top1)
        val_accs_top5.append(v_top5)
        val_f1_history.append(v_f1)

        print(f"Epoch {epoch+1}/{num_epochs} - Val Top-1: {v_top1:.2f}% | F1: {v_f1:.4f} | Breakdown: {acc_breakdown}")
        
        if v_top1 > best_val_acc:
            best_val_acc = v_top1
            torch.save(model.state_dict(), checkpoint_path)

    # Load best model for evaluation
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()

    v_loss, v_top1, v_top5, v_f1, cm, acc_breakdown, cm_df, misclassified, correct = validate_model(
            model, val_loader, criterion, device, num_classes=num_classes
        )

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, cmap="Blues", square=True, cbar=False)
    plt.title(f"{exp_name} - Final CM")
    plt.show()

    # Metrics Plot
    plt.figure(figsize=(18, 5))
    plt.subplot(1, 3, 1); plt.plot(train_losses, label='Train'); plt.plot(val_losses, label='Val', linestyle='--'); plt.title('Loss'); plt.legend()
    plt.subplot(1, 3, 2); plt.plot(train_accs, label='Train Top-1'); plt.plot(val_accs_top1, label='Val Top-1'); plt.title('Accuracy'); plt.legend()
    plt.subplot(1, 3, 3); plt.plot(val_f1_history, label='Val F1', color='purple'); plt.title('F1 Score'); plt.legend()
    plt.show()

    del model
    del optimizer
    torch.cuda.empty_cache()
    gc.collect()


## Setup DataLoaders

In [ ]:
baseline_train_set = DogDataset(train_df, TRAIN_DIR, transform=baseline_transform)
baseline_val_set = DogDataset(val_df, TRAIN_DIR, transform=baseline_transform)
augmented_train_set = DogDataset(train_df, TRAIN_DIR, transform=augmented_transform)
augmented_val_set = DogDataset(val_df, TRAIN_DIR, transform=augmented_transform)

batch_size = 32
loader_baseline_train = DataLoader(baseline_train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
loader_baseline_val = DataLoader(baseline_val_set, batch_size=batch_size, shuffle=False, pin_memory=True)
loader_augmented_train = DataLoader(augmented_train_set, batch_size=batch_size, shuffle=True, pin_memory=True)
loader_augmented_val = DataLoader(augmented_val_set, batch_size=batch_size, shuffle=False, pin_memory=True)


## Running the Experiments cleanly

In [ ]:
run_experiment("Baseline Frozen", "freeze", loader_baseline_train, loader_baseline_val, baseline_transform, num_epochs=20)
run_experiment("Augmented Frozen", "freeze", loader_augmented_train, loader_baseline_val, baseline_transform, num_epochs=20)
run_experiment("Baseline Train Augmented Val", "freeze", loader_baseline_train, loader_augmented_val, augmented_transform, num_epochs=20)
run_experiment("Augmented Train Augmented Val", "freeze", loader_augmented_train, loader_augmented_val, augmented_transform, num_epochs=20)
run_experiment("Augmented Train Finetune", "partial_finetune", loader_augmented_train, loader_baseline_val, baseline_transform, num_epochs=20)
run_experiment("Augmented Train Augmented Val Finetune", "partial_finetune", loader_augmented_train, loader_augmented_val, augmented_transform, num_epochs=20)
